# 0826_dongjin_029_sliding_shap_module
This module separates the real-time sliding window dynamic SHAP calculation code from experiment 028 into reusable functions (`calculate_sliding_shap` and `analyze_shap_convergence`).

It provides functionality to calculate and analyze dynamic SHAP values on streamed data windows to determine convergence and feature importance stability.


In [ ]:
import numpy as np
from collections import deque
import shap
from scipy.stats import kendalltau
import pandas as pd

def calculate_sliding_shap(bundle, inspection_type, X_raw, timestamps, window_sizes=[200, 500, 1000, 3000, 8000], snapshot_interval=50):
    """
    Computes dynamic sliding window SHAP values and tracks top features over time.
    
    Args:
        bundle (dict): The loaded model bundle containing preprocessors and models.
        inspection_type (int): The inspection type to evaluate.
        X_raw (pd.DataFrame): The raw feature data for the given inspection type.
        timestamps (pd.Series): The timestamps corresponding to X_raw.
        window_sizes (list of int): A list of window sizes to track.
        snapshot_interval (int): How frequently (in steps) to take a snapshot of the SHAP values.
        
    Returns:
        dict: A dictionary containing the history of timestamps, top features, and mean SHAP values for each window size.
    """
    # Load first ensemble model & preprocessor for this type for fast online TreeSHAP
    ckpt = bundle['ensemble_checkpoints'][0]
    model_info = bundle['members'][ckpt][inspection_type]
    preprocessor = model_info['preprocessor']
    model = model_info['model']
    explainer = shap.TreeExplainer(model)
    transformed_feature_names = preprocessor.get_feature_names_out()

    buffers = {w: deque(maxlen=w) for w in window_sizes}
    running_sum = {w: np.zeros(len(transformed_feature_names)) for w in window_sizes}

    history_timestamps = []
    history_top_features = {w: [] for w in window_sizes}
    history_shap_means = {w: [] for w in window_sizes}
    processed_count = 0

    # Transform features
    X_trans = preprocessor.transform(X_raw)
    if hasattr(X_trans, "toarray"):
        X_trans = X_trans.toarray()

    # Compute TreeSHAP
    shap_matrix = explainer.shap_values(X_trans)
    abs_shap_matrix = np.abs(shap_matrix)

    for i in range(len(abs_shap_matrix)):
        curr_shap = abs_shap_matrix[i]
        ts = timestamps.iloc[i]

        for w in window_sizes:
            buf = buffers[w]
            if len(buf) == w:
                old_shap = buf[0]
                running_sum[w] -= old_shap
            buf.append(curr_shap)
            running_sum[w] += curr_shap

            # Record periodic snapshot
            if processed_count % snapshot_interval == 0 and len(buf) > 0:
                mean_shap = running_sum[w] / len(buf)
                top_indices = np.argsort(-mean_shap)[:5]
                top_feats = [transformed_feature_names[idx] for idx in top_indices]
                history_top_features[w].append(top_feats)
                history_shap_means[w].append(mean_shap.copy())

        if processed_count % snapshot_interval == 0:
            history_timestamps.append(ts)
        processed_count += 1

    return {
        "history_timestamps": history_timestamps,
        "history_top_features": history_top_features,
        "history_shap_means": history_shap_means,
        "feature_names": transformed_feature_names,
        "window_sizes": window_sizes
    }

def analyze_shap_convergence(shap_results):
    """
    Analyzes the convergence and ranking stability of sliding SHAP values.
    
    Args:
        shap_results (dict): The output from calculate_sliding_shap().
        
    Returns:
        pd.DataFrame: A DataFrame containing convergence metrics for each window size.
    """
    convergence_results = []
    
    history_timestamps = shap_results["history_timestamps"]
    history_top_features = shap_results["history_top_features"]
    history_shap_means = shap_results["history_shap_means"]
    window_sizes = shap_results.get("window_sizes", list(history_shap_means.keys()))
    
    for w in window_sizes:
        if w not in history_shap_means:
            continue
            
        shap_hist = np.array(history_shap_means[w]) # Shape: (T, n_features)
        
        if len(shap_hist) == 0:
            continue

        # Kendall's Tau across successive time steps
        taus = []
        for t in range(len(shap_hist) - 1):
            tau, _ = kendalltau(shap_hist[t], shap_hist[t+1])
            if not np.isnan(tau):
                taus.append(tau)

        # Number of times #1 feature changed (Ranking Churn)
        top1_series = [top_list[0] for top_list in history_top_features[w]]
        churn_count = sum(1 for i in range(len(top1_series)-1) if top1_series[i] != top1_series[i+1])
        churn_rate = churn_count / (len(top1_series) - 1) * 100 if len(top1_series) > 1 else 0.0

        mean_var = np.mean(np.var(shap_hist, axis=0)) if len(shap_hist) > 0 else 0
        mean_tau = np.mean(taus) if len(taus) > 0 else 1.0

        convergence_results.append({
            "Window Size (W)": w,
            "Expected Defects": f"{w * 0.012:.1f}",
            "Kendall Tau (Stability)": mean_tau,
            "SHAP Variance (x1e-3)": mean_var * 1000,
            "#1 Feature Churn Rate (%)": churn_rate,
            "Convergence Status": "Converged (Stable)" if mean_tau > 0.95 else ("Partial" if mean_tau > 0.85 else "Unconverged (Noisy)")
        })

    return pd.DataFrame(convergence_results)
